In [15]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Session09Part01")
    .master("local[*]")
    .getOrCreate()
)

In [16]:
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("request_count", IntegerType(), True),
    StructField("error_count", IntegerType(), True),
    StructField("latency_ms", DoubleType(), True),
    StructField("bytes_in", DoubleType(), True),
    StructField("bytes_out", DoubleType(), True),
])

In [17]:
events_path = "../datasets/service_events.csv"

In [18]:
events_df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(events_path)
)

In [19]:
events_df.printSchema()
events_df.show(5, truncate=False)

print("Rows:", events_df.count())
print("Columns:", events_df.columns)

root
 |-- event_id: integer (nullable = true)
 |-- service: string (nullable = true)
 |-- region: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- request_count: integer (nullable = true)
 |-- error_count: integer (nullable = true)
 |-- latency_ms: double (nullable = true)
 |-- bytes_in: double (nullable = true)
 |-- bytes_out: double (nullable = true)

+--------+---------------+--------+-------------------+-------------+-----------+----------+--------+---------+
|event_id|service        |region  |event_time         |request_count|error_count|latency_ms|bytes_in|bytes_out|
+--------+---------------+--------+-------------------+-------------+-----------+----------+--------+---------+
|1       |auth           |eu-west |2026-05-04 00:00:00|1240         |8          |82.4      |1.45E7  |3.82E7   |
|2       |payments       |eu-west |2026-05-04 00:00:00|860          |21         |146.2     |1.92E7  |4.41E7   |
|3       |search         |us-east |2026-05-04 01:00:00|211

In [20]:
events_df.createOrReplaceTempView("service_events")

In [21]:
spark.sql("""
    SELECT service, region, event_time, request_count
    FROM service_events
    ORDER BY event_time
    LIMIT 10
""").show(truncate=False)

+---------------+--------+-------------------+-------------+
|service        |region  |event_time         |request_count|
+---------------+--------+-------------------+-------------+
|auth           |eu-west |2026-05-04 00:00:00|1240         |
|payments       |eu-west |2026-05-04 00:00:00|860          |
|search         |us-east |2026-05-04 01:00:00|2110         |
|recommendations|us-east |2026-05-04 01:00:00|1680         |
|auth           |ap-south|2026-05-04 02:00:00|980          |
|payments       |ap-south|2026-05-04 02:00:00|740          |
|search         |eu-west |2026-05-04 03:00:00|2320         |
|recommendations|eu-west |2026-05-04 03:00:00|1760         |
|auth           |us-east |2026-05-04 09:00:00|1480         |
|payments       |us-east |2026-05-04 09:00:00|1120         |
+---------------+--------+-------------------+-------------+



In [22]:
spark.sql("""
    SELECT service, region, event_time, (error_count/request_count)*100 AS error_rate
    FROM service_events
    ORDER BY event_time
    LIMIT 10
""").show(truncate=False)

+---------------+--------+-------------------+-------------------+
|service        |region  |event_time         |error_rate         |
+---------------+--------+-------------------+-------------------+
|auth           |eu-west |2026-05-04 00:00:00|0.6451612903225806 |
|payments       |eu-west |2026-05-04 00:00:00|2.441860465116279  |
|search         |us-east |2026-05-04 01:00:00|0.6161137440758294 |
|recommendations|us-east |2026-05-04 01:00:00|1.0714285714285714 |
|auth           |ap-south|2026-05-04 02:00:00|0.40816326530612246|
|payments       |ap-south|2026-05-04 02:00:00|2.2972972972972974 |
|search         |eu-west |2026-05-04 03:00:00|0.646551724137931  |
|recommendations|eu-west |2026-05-04 03:00:00|1.3636363636363635 |
|auth           |us-east |2026-05-04 09:00:00|0.40540540540540543|
|payments       |us-east |2026-05-04 09:00:00|2.3214285714285716 |
+---------------+--------+-------------------+-------------------+



In [25]:
spark.sql("""
    SELECT region, SUM(request_count) AS requests_per_region
    FROM service_events
    GROUP BY region
    LIMIT 10
""").show(truncate=False)

+--------+-------------------+
|region  |requests_per_region|
+--------+-------------------+
|eu-west |12630              |
|us-east |13040              |
|ap-south|10460              |
+--------+-------------------+



In [27]:
spark.sql("""
    SELECT service, SUM(request_count) AS requests_per_service
    FROM service_events
    GROUP BY service
    LIMIT 10
""").show(truncate=False)

+---------------+--------------------+
|service        |requests_per_service|
+---------------+--------------------+
|auth           |7580                |
|recommendations|9880                |
|payments       |5590                |
|search         |13080               |
+---------------+--------------------+



In [28]:
spark.stop()